# Brain Age Prediction — End-to-End 3D ResNet

Replaces the VGG16 feature extraction + SVR pipeline with a **3D ResNet-18** trained end-to-end for age regression.

**Runtime:** GPU (T4 or better). Go to *Runtime → Change runtime type → GPU*.

**Pipeline:**
```
NIfTI files (3D MRI)
      ↓  resize to 96×96×96, z-score normalize
3D ResNet-18  (end-to-end, regression head)
      ↓
Predicted Age  →  MAE / RMSE / Pearson r
```

## 1. Install Dependencies

In [ ]:
# monai provides medical-imaging utilities (NIfTI transforms, etc.)
!pip install -q monai nibabel openpyxl
import importlib, sys
print('torch:', __import__('torch').__version__)
print('monai:', __import__('monai').__version__)

## 2. Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── EDIT THESE PATHS TO MATCH YOUR GOOGLE DRIVE LAYOUT ──────────────────────
NII_DIR        = '/content/drive/MyDrive/BrainAge/nifti'      # folder of .nii / .nii.gz files
DEMOGRAPHICS   = '/content/drive/MyDrive/BrainAge/demographics.xlsx'  # Excel with Subject ID + Age columns
CHECKPOINT_DIR = '/content/drive/MyDrive/BrainAge/checkpoints'        # saved model weights go here
# ─────────────────────────────────────────────────────────────────────────────

# Column names inside the Excel file
SUBJECT_COL = 'RID'   # subject / research ID column
AGE_COL     = 'AGE'   # chronological age column

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint directory:', CHECKPOINT_DIR)

## 3. Imports

In [ ]:
import os, glob, time, random
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import monai.transforms as T

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0),
          f'  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Reproducibility Seed

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

## 5. Data Loading & Train/Test Split

In [ ]:
import re

# ── Load demographics ──────────────────────────────────────────────────────
demo = pd.read_excel(DEMOGRAPHICS)
demo[SUBJECT_COL] = demo[SUBJECT_COL].astype(str).str.strip()
demo[AGE_COL]     = pd.to_numeric(demo[AGE_COL], errors='coerce')
demo = demo.dropna(subset=[SUBJECT_COL, AGE_COL]).reset_index(drop=True)
print(f'Demographics loaded: {len(demo)} subjects')

# ── Match NIfTI files to subjects ─────────────────────────────────────────
all_nii = glob.glob(os.path.join(NII_DIR, '**', '*.nii*'), recursive=True)
print(f'NIfTI files found: {len(all_nii)}')

records = []
for _, row in demo.iterrows():
    rid = row[SUBJECT_COL]
    age = float(row[AGE_COL])
    # Use word-boundary regex so RID "12" does not match "120", "121", etc.
    pattern = re.compile(r'(?<!\d)' + re.escape(rid) + r'(?!\d)')
    matches = [f for f in all_nii if pattern.search(os.path.basename(f))]
    if matches:
        records.append({'rid': rid, 'age': age, 'path': matches[0]})

df = pd.DataFrame(records).sort_values('age').reset_index(drop=True)
print(f'Matched subjects: {len(df)}')
print(df[['rid','age']].describe())

## 5b. Dataset Cleaning

Loads every matched NIfTI once and drops files that would crash the training loop:
- Unreadable / corrupt file
- Fewer than 3 spatial dimensions
- Any spatial dimension smaller than 32 voxels
- Blank scan (zero variance)
- NaN or Inf values in image data

In [ ]:
def validate_nifti(path: str, min_dim: int = 32) -> tuple[bool, str]:
    """
    Tries to load a NIfTI file and checks for common data quality issues.

    Returns:
        (True, '')           — file is clean
        (False, reason_str)  — file should be dropped
    """
    try:
        img  = nib.load(path)
        data = img.get_fdata(dtype=np.float32)
    except Exception as e:
        return False, f'load error: {e}'

    # Must be at least 3D
    if data.ndim < 3:
        return False, f'only {data.ndim}D (need >= 3D)'

    # Each spatial dimension must be large enough to survive resizing
    for ax, sz in enumerate(data.shape[:3]):
        if sz < min_dim:
            return False, f'dim {ax} has size {sz} (min {min_dim})'

    # No NaN / Inf
    if not np.isfinite(data).all():
        n_bad = int(np.sum(~np.isfinite(data)))
        return False, f'{n_bad} NaN/Inf voxels'

    # Blank scan — zero or near-zero variance
    if data.std() < 1e-6:
        return False, 'near-zero variance (blank scan)'

    return True, ''


print(f'Validating {len(df)} NIfTI files ...')
valid_mask = []
dropped    = []

for _, row in df.iterrows():
    ok, reason = validate_nifti(row['path'])
    valid_mask.append(ok)
    if not ok:
        dropped.append({'rid': row['rid'], 'path': row['path'], 'reason': reason})

df = df[valid_mask].reset_index(drop=True)

print(f'Passed:  {sum(valid_mask)}')
print(f'Dropped: {len(dropped)}')
if dropped:
    print('\nDropped subjects:')
    for d in dropped:
        print(f"  RID {d['rid']:>10}  —  {d['reason']}")
        print(f"               {d['path']}")

assert len(df) >= 4, 'Too few valid subjects to train. Check your NII_DIR path and file names.'

In [ ]:
# ── Train / Val / Test split — 70 / 15 / 15 % ──────────────────────────
# Sort by age, then assign systematically using a cyclic rule over blocks of 20:
#   positions {3, 10, 17} → val  (~15%)
#   positions {6, 13, 19} → test (~15%)
#   all others            → train (~70%)
_CYCLE       = 20
_VAL_OFFSETS  = {3, 10, 17}
_TEST_OFFSETS = {6, 13, 19}

val_mask   = [(i % _CYCLE) in _VAL_OFFSETS  for i in range(len(df))]
test_mask  = [(i % _CYCLE) in _TEST_OFFSETS for i in range(len(df))]
train_mask = [not v and not t for v, t in zip(val_mask, test_mask)]

df_train = df[train_mask].reset_index(drop=True)
df_val   = df[val_mask].reset_index(drop=True)
df_test  = df[test_mask].reset_index(drop=True)

n_total = len(df)
print(f'Total subjects: {n_total}')
print(f'Train: {len(df_train)} ({100*len(df_train)/n_total:.0f}%)  |  '
      f'Val: {len(df_val)} ({100*len(df_val)/n_total:.0f}%)  |  '
      f'Test (held out): {len(df_test)} ({100*len(df_test)/n_total:.0f}%)')
print(f'Train age range: [{df_train.age.min():.1f}, {df_train.age.max():.1f}]')
print(f'Val   age range: [{df_val.age.min():.1f}, {df_val.age.max():.1f}]')
print(f'Test  age range: [{df_test.age.min():.1f}, {df_test.age.max():.1f}]')

## 6. Dataset & Transforms

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE      = (96, 96, 96)   # spatial dimensions after resize (D, H, W)
BATCH_SIZE    = 2              # keep low for Colab T4 (15 GB VRAM)
NUM_EPOCHS    = 100
WARMUP_EPOCHS = 5              # linear LR warmup before cosine annealing
LR            = 1e-5           # reduced 10x to prevent early val-loss spikes
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2

In [ ]:
# ── MONAI transforms ───────────────────────────────────────────────────────
# LoadImage → add channel dim → resize → intensity normalize

train_transforms = T.Compose([
    T.LoadImaged(keys=['image']),
    T.EnsureChannelFirstd(keys=['image']),
    T.Orientationd(keys=['image'], axcodes='RAS'),
    T.Spacingd(keys=['image'], pixdim=(2.0, 2.0, 2.0), mode='bilinear'),
    T.ResizeWithPadOrCropd(keys=['image'], spatial_size=IMG_SIZE),
    T.NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    # Augmentation
    T.RandFlipd(keys=['image'], prob=0.5, spatial_axis=0),
    T.RandFlipd(keys=['image'], prob=0.5, spatial_axis=1),
    T.RandFlipd(keys=['image'], prob=0.5, spatial_axis=2),
    T.RandScaleIntensityd(keys=['image'], factors=0.1, prob=0.5),
    T.RandShiftIntensityd(keys=['image'], offsets=0.1, prob=0.5),
    T.ToTensord(keys=['image']),
])

val_transforms = T.Compose([
    T.LoadImaged(keys=['image']),
    T.EnsureChannelFirstd(keys=['image']),
    T.Orientationd(keys=['image'], axcodes='RAS'),
    T.Spacingd(keys=['image'], pixdim=(2.0, 2.0, 2.0), mode='bilinear'),
    T.ResizeWithPadOrCropd(keys=['image'], spatial_size=IMG_SIZE),
    T.NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    T.ToTensord(keys=['image']),
])

In [ ]:
class BrainAgeDataset(Dataset):
    """
    Wraps a DataFrame with columns [path, age] and applies MONAI transforms.
    Returns (image_tensor [1, D, H, W], age_scalar).
    """
    def __init__(self, df: pd.DataFrame, transforms):
        self.data       = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row  = self.data.iloc[idx]
        item = self.transforms({'image': row['path']})
        img  = item['image'].float()          # [1, D, H, W]
        age  = torch.tensor(row['age'], dtype=torch.float32)
        return img, age


train_ds = BrainAgeDataset(df_train, train_transforms)
val_ds   = BrainAgeDataset(df_val,   val_transforms)
test_ds  = BrainAgeDataset(df_test,  val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}')

# Sanity-check one sample
img_sample, age_sample = train_ds[0]
print(f'Image tensor shape: {img_sample.shape}  |  Age: {age_sample.item():.1f}')

## 7. 3D ResNet-18 Architecture

In [ ]:
# ── 3D BasicBlock ──────────────────────────────────────────────────────────
class BasicBlock3D(nn.Module):
    expansion = 1

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm3d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


# ── 3D ResNet-18 ───────────────────────────────────────────────────────────
class ResNet3D(nn.Module):
    """
    3D ResNet-18 for single-output regression (brain age).

    Architecture:
        Stem:    7×7×7 conv, stride 2  → 64 ch  → MaxPool stride 2
        Layer1:  2 × BasicBlock(64)
        Layer2:  2 × BasicBlock(128),  stride 2
        Layer3:  2 × BasicBlock(256),  stride 2
        Layer4:  2 × BasicBlock(512),  stride 2
        Head:    AdaptiveAvgPool → Dropout → FC(512 → 1)
    """
    def __init__(self, in_channels: int = 1, dropout: float = 0.3):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=3, stride=2, padding=1),
        )

        # Residual stages
        self.layer1 = self._make_layer(64,  64,  blocks=2, stride=1)
        self.layer2 = self._make_layer(64,  128, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=2, stride=2)

        # Regression head
        self.pool    = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(512, 1)

        self._init_weights()

    @staticmethod
    def _make_layer(in_ch: int, out_ch: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [BasicBlock3D(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock3D(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)   # [B, 512]
        x = self.dropout(x)
        return self.fc(x).squeeze(1)  # [B]


model = ResNet3D(in_channels=1, dropout=0.3).to(DEVICE)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {train_params:,}')

# Dry-run to verify shapes
with torch.no_grad():
    dummy = torch.zeros(1, 1, *IMG_SIZE, device=DEVICE)
    out   = model(dummy)
    print(f'Output shape for dummy input {tuple(dummy.shape)}: {tuple(out.shape)}')

## 8. Training Setup

In [ ]:
import math

criterion = nn.HuberLoss(delta=1.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Linear warmup for WARMUP_EPOCHS, then cosine decay to eta_min=1e-6
def _lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    eta_min_factor = 1e-6 / LR
    return eta_min_factor + (1.0 - eta_min_factor) * cosine_decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

# Mixed-precision scaler (speeds up training on T4)
scaler = GradScaler()

## 9. Training Loop

In [ ]:
BEST_CKPT = os.path.join(CHECKPOINT_DIR, 'best_resnet3d.pth')

history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_r': []}

best_val_mae = float('inf')


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.set_grad_enabled(train):
        for imgs, ages in loader:
            imgs  = imgs.to(DEVICE,  non_blocking=True)
            ages  = ages.to(DEVICE,  non_blocking=True)

            if train:
                optimizer.zero_grad()
                with autocast():
                    preds = model(imgs)
                    loss  = criterion(preds, ages)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                with autocast():
                    preds = model(imgs)
                    loss  = criterion(preds, ages)

            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(ages.detach().cpu().numpy())

    n          = len(all_targets)
    avg_loss   = total_loss / n
    mae_val    = mean_absolute_error(all_targets, all_preds)
    rmse_val   = np.sqrt(mean_squared_error(all_targets, all_preds))
    r_val, _   = pearsonr(all_targets, all_preds) if n >= 2 else (float('nan'), None)
    return avg_loss, mae_val, rmse_val, r_val


print(f'{'Epoch':>6}  {'Train Loss':>11}  {'Val MAE':>9}  {'Val RMSE':>9}  {'Val r':>7}  {'LR':>8}  Time')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss, *_           = run_epoch(train_loader, train=True)
    val_loss, v_mae, v_rmse, v_r = run_epoch(val_loader, train=False)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(v_mae)
    history['val_rmse'].append(v_rmse)
    history['val_r'].append(v_r)

    elapsed = time.time() - t0
    print(f'{epoch:>6}  {train_loss:>11.4f}  {v_mae:>9.4f}  {v_rmse:>9.4f}  {v_r:>7.4f}  {current_lr:>8.2e}  {elapsed:.0f}s')

    # Save best model
    if v_mae < best_val_mae:
        best_val_mae = v_mae
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_mae': v_mae}, BEST_CKPT)
        print(f'  ✓ Best model saved (MAE={v_mae:.4f})')

print(f'\nTraining complete. Best Val MAE: {best_val_mae:.4f}')

## 10. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train Loss')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss')
axes[0].set_title('L1 Loss (MAE)')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, history['val_mae'],  label='Val MAE',  color='tab:orange')
axes[1].plot(epochs, history['val_rmse'], label='Val RMSE', color='tab:red')
axes[1].set_title('Regression Metrics')
axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs, history['val_r'], label='Pearson r', color='tab:green')
axes[2].set_title('Pearson Correlation')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
os.makedirs('Results', exist_ok=True)
plt.savefig('Results/brain_age_resnet3d_training_curves.png', dpi=150)
plt.show()

## 11. Final Evaluation on Validation Set

In [ ]:
# Load the best checkpoint
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
best_epoch = ckpt['epoch']
print(f'Loaded best model from epoch {best_epoch} (Val MAE = {ckpt["val_mae"]:.4f})')

# ── Step 1: collect val predictions to fit the bias corrector ─────────────
model.eval()
val_preds, val_ages = [], []

with torch.no_grad():
    for imgs, ages in val_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            preds = model(imgs)
        val_preds.extend(preds.cpu().numpy())
        val_ages.extend(ages.numpy())

val_ages  = np.array(val_ages)
val_preds = np.array(val_preds)

val_mae_uncorr  = mean_absolute_error(val_ages, val_preds)
val_rmse_uncorr = np.sqrt(mean_squared_error(val_ages, val_preds))
val_r_uncorr, val_p = pearsonr(val_ages, val_preds)

print('\n── Validation Set (Uncorrected) ────────────────────')
print(f'  MAE:       {val_mae_uncorr:.4f} years')
print(f'  RMSE:      {val_rmse_uncorr:.4f} years')
print(f'  Pearson r: {val_r_uncorr:.4f}  (p={val_p:.2e})')

# ── Step 2: fit bias corrector on val predictions ──────────────────────────
# Maps raw_pred → true_age to undo regression-to-the-mean.
# Expected: slope > 1 (stretches predictions toward extremes), intercept ≠ 0.
from sklearn.linear_model import LinearRegression

bias_corrector = LinearRegression()
bias_corrector.fit(val_preds.reshape(-1, 1), val_ages)
slope     = bias_corrector.coef_[0]
intercept = bias_corrector.intercept_
print(f'\n── Bias Corrector (fitted on val) ──────────────────')
print(f'  slope={slope:.4f}  intercept={intercept:.4f}')
if abs(slope - 1.0) < 0.05:
    print('  WARNING: slope ≈ 1 — correction is doing very little. '
          'Model may already be well-calibrated, or training needs more epochs.')

# ── Step 3: run on TEST set and apply corrector ────────────────────────────
test_preds_raw, test_ages_all = [], []

with torch.no_grad():
    for imgs, ages in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            preds = model(imgs)
        test_preds_raw.extend(preds.cpu().numpy())
        test_ages_all.extend(ages.numpy())

test_ages_all   = np.array(test_ages_all)
test_preds_raw  = np.array(test_preds_raw)
test_preds_corr = bias_corrector.predict(test_preds_raw.reshape(-1, 1))

test_mae_uncorr  = mean_absolute_error(test_ages_all, test_preds_raw)
test_mae_corr    = mean_absolute_error(test_ages_all, test_preds_corr)
test_rmse_corr   = np.sqrt(mean_squared_error(test_ages_all, test_preds_corr))
test_r_corr, _   = pearsonr(test_ages_all, test_preds_corr)

print('\n── Test Set Results ────────────────────────────────')
print(f'  Best model epoch:    {best_epoch}')
print(f'  Uncorrected MAE:     {test_mae_uncorr:.4f} years')
print(f'  Corrected MAE:       {test_mae_corr:.4f} years')
print(f'  Corrected RMSE:      {test_rmse_corr:.4f} years')
print(f'  Corrected Pearson r: {test_r_corr:.4f}')
print(f'  Bias correction:     slope={slope:.4f}, intercept={intercept:.4f}')

# keep val vars accessible for scatter plot
all_ages  = val_ages
all_preds = val_preds

## 12. Predicted vs True Age Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(all_ages, all_preds, alpha=0.6, edgecolors='k', linewidths=0.4, s=60)

# Perfect-prediction diagonal
lim = [min(all_ages.min(), all_preds.min()) - 2,
       max(all_ages.max(), all_preds.max()) + 2]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')

# Regression line
m, b = np.polyfit(all_ages, all_preds, 1)
ax.plot(lim, [m * v + b for v in lim], 'b-', linewidth=1.5, label='Regression line')

ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('True Age (years)', fontsize=13)
ax.set_ylabel('Predicted Age (years)', fontsize=13)
ax.set_title('Brain Age Prediction — 3D ResNet-18', fontsize=14)
ax.legend(fontsize=11)
ax.text(0.05, 0.95,
        f'MAE  = {val_mae_final:.2f} yr\nRMSE = {val_rmse_final:.2f} yr\nr    = {val_r_final:.3f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
os.makedirs('Results', exist_ok=True)
plt.savefig('Results/brain_age_resnet3d_results.png', dpi=150)
plt.show()

## 13. Save Predictions to CSV

In [ ]:
os.makedirs('Results', exist_ok=True)

results_df = pd.DataFrame({
    'RID':            df_val['rid'].values,
    'True_Age':       all_ages,
    'Predicted_Age':  all_preds,
    'Brain_Age_Gap':  all_preds - all_ages,
    'Split':          'validation',
})

out_csv = 'Results/brain_age_resnet3d_predictions.csv'
results_df.to_csv(out_csv, index=False)
print(f'Saved predictions to: {out_csv}')
results_df.head(10)

## 14. Single-Subject Inference (Optional)
Run this cell to predict brain age for a single new NIfTI file.

In [ ]:
NEW_NII_PATH = '/content/drive/MyDrive/BrainAge/new_subject.nii.gz'  # ← change this

infer_transform = val_transforms  # reuse val (no augmentation)

item = infer_transform({'image': NEW_NII_PATH})
img  = item['image'].float().unsqueeze(0).to(DEVICE)  # add batch dim

model.eval()
with torch.no_grad():
    predicted_age = model(img).item()

print(f'Predicted brain age: {predicted_age:.1f} years')